# Tool 1 — Policy Search: manual test notebook

Exercises the hybrid retrieval + LLM answer pipeline in `src/tools/policy_search.py`.

**Note:** cells (a), (b), and (e) work with no API key (routing + retrieval only).
Cells (c) and (d) call the LLM, so they need a **valid** key in `.env` for the
active provider (`GOOGLE_API_KEY` for gemini). With the placeholder key they will
raise `API_KEY_INVALID`.

Run top-to-bottom. Every cell prints actual chunk text / scores, not just pass/fail.

### (a) Imports + config — which provider/model is in use

In [ ]:
# testing the llm call — which LLM is actually working right now?
# Makes one tiny real call to the active provider in config.json and reports it.
import os, sys
sys.path.insert(0, os.path.abspath(".."))
from src.config import CONFIG
import src.tools.policy_search as ps

_prov = CONFIG["active_provider"]
_model = CONFIG["providers"][_prov].get("model")
try:
    _reply = ps._GENERATE("Reply with exactly the word: OK")
    print(f"✅ LLM WORKING  ->  {_prov} / {_model}")
    print(f"   response: {_reply[:100]!r}")
except Exception as _e:
    print(f"❌ LLM NOT WORKING  ->  {_prov} / {_model}")
    print(f"   {type(_e).__name__}: {_e}")

In [ ]:
import os, sys

# Make `import src...` work when the kernel's cwd is notebooks/.
sys.path.insert(0, os.path.abspath(".."))

from src.config import CONFIG
import src.tools.policy_search as ps

prov = CONFIG["active_provider"]
pconf = CONFIG["providers"][prov]
print(f"Active provider : {prov}")
print(f"Model           : {pconf.get('model')}")
print(f"Embedding model : {CONFIG['embedding_model']}")
print(f"Retrieval       : top_k={CONFIG['retrieval']['top_k_chunks']}, rrf_k={CONFIG['retrieval']['rrf_k']}")
print(f"Chunking        : size={CONFIG['chunking']['chunk_size']}, overlap={CONFIG['chunking']['chunk_overlap']}")
print(f"Total chunks    : {len(ps.CHUNKS)}")

key_env = pconf.get("api_key_env")
if key_env:
    val = os.environ.get(key_env, "")
    looks_placeholder = (not val) or val.lower().startswith("your")
    print(f"{key_env:16}: {'MISSING/PLACEHOLDER - LLM cells (c,d) will fail' if looks_placeholder else 'set'}")

### (b) Routing only — `route_query_to_call_type()` on 6 queries (5 routable + 1 nonsense)
No LLM, no retrieval — just the keyword router. `None` = no match (will search all docs).

In [ ]:
routing_queries = [
    "What is the replacement policy?",          # -> replacement
    "How long does a repair take?",             # -> repair
    "What happens if I return a damaged item?", # -> logistics
    "Can you set up my device remotely?",       # -> device
    "I want to escalate this to a manager",     # -> escalation
    "asdkjhasd random nonsense query",          # -> None (no match)
]
for q in routing_queries:
    ct = ps.route_query_to_call_type(q)
    print(f"  {q!r:48} -> {ct}")

### (c) End-to-end `search_policy()` on the 5 test queries — full generated answer
**Requires a valid API key.** Each query routes -> BM25+FAISS -> RRF -> LLM answer.
The nonsense query should trigger the "policy documents do not cover this" fallback.

In [ ]:
test_queries = [
    "What is the replacement policy?",
    "How long does a repair take?",
    "What happens if I return a damaged item?",
    "Can you set up my device remotely?",
    "asdkjhasd random nonsense query",
]
for q in test_queries:
    print("=" * 88)
    print("Q:", q)
    print("A:", ps.search_policy(q))

### (d) `search_policy_with_context()` with an explicit `call_type` — raw chunks vs answer
Passing `call_type="repair"` skips routing. Prints the raw retrieved chunks (`chunks_used`)
separately from the generated answer so you can compare retrieval vs generation.
**Requires a valid API key** for the answer half.

In [ ]:
result = ps.search_policy_with_context("How long does a repair take?", call_type="repair")

print("routed_call_type:", result["routed_call_type"])
print("\n--- RAW RETRIEVED CHUNKS (chunks_used) ---")
for i, ch in enumerate(result["chunks_used"], 1):
    print(f"[{i}] {ch}\n")
print("--- GENERATED ANSWER ---")
print(result["answer"])

### (e) Effect of `chunk_size` on retrieval — mutate config + reindex, no restart, no LLM
Edits `CONFIG["chunking"]["chunk_size"]` in memory, calls `ps.rebuild_indexes()`, and
reruns retrieval for one query. Prints chunk count + fused chunk text/scores before and
after, then restores the original so later cells behave normally.

In [ ]:
def show_retrieval(query, call_type):
    """Run the hybrid retriever (prints BM25/FAISS/ensemble dev logs) and show fused docs."""
    docs = ps._retrieve(query, call_type)
    print(f"  -> {len(docs)} fused docs returned:")
    for rank, d in enumerate(docs, 1):
        print(f"     #{rank} id={d.metadata['chunk_id']} [{d.metadata['source_file']}] {d.page_content[:80]!r}")

q = "How long does a repair take?"

print(f"===== BEFORE  chunk_size={ps.CONFIG['chunking']['chunk_size']} overlap={ps.CONFIG['chunking']['chunk_overlap']}  (total chunks={len(ps.CHUNKS)}) =====")
show_retrieval(q, "repair")

# Mutate config in-memory and force a reindex (rebuilds FAISS + BM25 retrievers).
ps.CONFIG["chunking"]["chunk_size"] = 120
ps.CONFIG["chunking"]["chunk_overlap"] = 20
ps.rebuild_indexes()

print(f"\n===== AFTER   chunk_size={ps.CONFIG['chunking']['chunk_size']} overlap={ps.CONFIG['chunking']['chunk_overlap']}  (total chunks={len(ps.CHUNKS)}) =====")
show_retrieval(q, "repair")

# Restore defaults so cells (c)/(d) and reruns use the original chunking.
ps.CONFIG["chunking"]["chunk_size"] = 300
ps.CONFIG["chunking"]["chunk_overlap"] = 50
ps.rebuild_indexes()
print(f"\nRESTORED chunk_size={ps.CONFIG['chunking']['chunk_size']} (total chunks={len(ps.CHUNKS)})")